# Discovery Representation Comparison, Validation, And Inspection

This notebook restores one trained checkpoint, builds one shared discovery window set for one decode task, and compares three representation arms on the exact same selected windows: `baseline`, `whitening_only`, and `whitening_plus_held_out_alignment`.

The primary benchmark is a within-session held-out comparison shared across all three arms. Standard cross-session test validation remains visible as a secondary metric for `baseline` and `whitening_only`.

In [ ]:
# Configuration - edit these values, then Run All
# Select the training run first, then the decode task, shared discovery budgets, and comparison controls.
# Verified against AllenSDK Visual Behavior Neuropixels tables: use the major binary task labels below.
# Recommended starting points for this project:
#   'stimulus_change'  -> recommended alias for AllenSDK change events ('stimulus_presentations.is_change' or 'trials.is_change')
#   'trials.go'        -> real change trials
#   'trials.catch'     -> sham-change trials
#   'trials.hit'       -> rewarded lick after a change
#   'trials.miss'      -> missed response on a change trial
#   'trials.false_alarm'
#   'trials.correct_reject'
#   'stimulus_presentations.omitted'  -> omitted flashes (useful if you want omission-related motifs)
# Advanced aliases / custom paths:
#   'stimulus_presentations.is_change', 'trials.is_change'

DECODE_TYPE = 'stimulus_change'
TRAINING_RUN_ID = None  # None = use the latest exported training run; or set a specific run_id like 'run_20260407_153000'
DISCOVERY_DEVICE_MODE = 'auto'  # 'auto' = GPU encoder if available, 'cpu' = force CPU only
STAGE_PREPARED_SESSIONS_LOCALLY = True  # recommended in Colab; copies just the discovery/test .h5 files off Drive before long runs
SESSION_HOLDOUT_FRACTION = 0.5  # fraction of each valid session's selected discovery windows reserved for the primary within-session held-out benchmark
SESSION_HOLDOUT_SEED = None  # None = reuse VALIDATION_SHUFFLE_SEED if set, else the restored experiment seed
RUN_STANDARD_TEST_VALIDATION = True  # also report the existing cross-session test metrics for baseline and whitening_only
INSPECT_ARM = 'whitening_only'  # one of: 'baseline', 'whitening_only', 'whitening_plus_held_out_alignment'

# Shared discovery overrides across all three arms - None = reuse the saved training/runtime config value.
DISCOVERY_MAX_BATCHES = 128  # examples: 8 for fast debug runs, 16-32 for deeper discovery scans
DISCOVERY_TOP_K_CANDIDATES = None  # examples: 32, 64, 128
DISCOVERY_CANDIDATE_SESSION_BALANCE_FRACTION = None  # examples: 0.2 to reduce single-session dominance, 1.0 to disable balancing
DISCOVERY_MIN_CANDIDATE_SCORE = None  # lower to admit more candidates, e.g. -0.1 or 0.0
DISCOVERY_MIN_CLUSTER_SIZE = None  # examples: 2 for minimal clustering, 3-4 for stricter motif groups
DISCOVERY_PROBE_EPOCHS = None  # examples: 25 for debug, 50-100 for longer probe fitting
DISCOVERY_PROBE_LEARNING_RATE = None  # examples: 1.0e-2, 5.0e-3
VALIDATION_MAX_BATCHES = 128  # maps to evaluation.max_batches; examples: 8, 16, 32
VALIDATION_SHUFFLE_SEED = None  # maps to discovery.shuffle_seed; change only to alter the shuffle control

NOTEBOOK_STEP_LOG_EVERY = 16  # print a notebook log line every N steps from the CLI wrapper

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil
import json
import pandas as pd
from datetime import datetime

drive.mount('/content/drive')
repo_root = Path('/content/predictive_circuit_coding')
github_repo_url = 'https://github.com/jacobposchl/predictive_circuit_coding'

# Shared folder from the storage-heavy Drive account: read dataset from here.
shared_drive_root = Path('/content/drive/MyDrive/pcc_runs')
drive_data_root = shared_drive_root / 'data' / 'allen_visual_behavior_neuropixels'

# Keep exported Colab outputs in a clearly different folder name so they do not get
# confused with the source dataset bundle.
drive_export_root = Path('/content/drive/MyDrive/pcc_colab_outputs')
drive_export_root.mkdir(parents=True, exist_ok=True)

print('Repo root:', repo_root)
print('GitHub repo:', github_repo_url)
print('Shared dataset root:', drive_data_root)
print('Drive export root:', drive_export_root)

In [ ]:
if not repo_root.exists():
    !git clone {github_repo_url} {repo_root}
%cd {repo_root}
!git pull --ff-only
# Keep Colab installs minimal to avoid restart prompts from preloaded widget packages.
!pip install -e .

In [ ]:
repo_dataset_root = repo_root / 'data' / 'allen_visual_behavior_neuropixels'
repo_artifacts_root = repo_root / 'artifacts'

assert drive_data_root.exists(), f'Missing Drive dataset bundle: {drive_data_root}'

repo_dataset_root.parent.mkdir(parents=True, exist_ok=True)
if repo_dataset_root.exists() or repo_dataset_root.is_symlink():
    if repo_dataset_root.is_symlink():
        repo_dataset_root.unlink()
    else:
        shutil.rmtree(repo_dataset_root)
repo_dataset_root.symlink_to(drive_data_root, target_is_directory=True)

if not repo_artifacts_root.exists():
    repo_artifacts_root.mkdir(parents=True, exist_ok=True)

print('Linked dataset into repo:', repo_dataset_root)
print('Using local artifact directory:', repo_artifacts_root)

In [ ]:
import yaml
from predictive_circuit_coding.data import load_split_manifest
from predictive_circuit_coding.utils import (
    NotebookStageReporter,
    build_notebook_discovery_comparison_paths,
    build_notebook_discovery_export_path,
    build_notebook_discovery_runtime_config,
    describe_notebook_compute_targets,
    export_notebook_discovery_comparison_artifacts,
    load_notebook_split_counts,
    materialize_notebook_prepared_sessions,
    resolve_notebook_checkpoint,
    restore_latest_exported_artifacts,
    run_notebook_discovery_representation_comparison,
    verify_paths_exist,
)

reporter = NotebookStageReporter(name='colab-discover-compare', expected_duration='5-20 minutes depending on discovery budgets')
reporter.banner('Discovery Representation Comparison', subtitle='Baseline vs whitening vs whitening + held-out alignment on one shared decode task')

data_config = repo_root / 'configs' / 'pcc' / 'allen_visual_behavior_neuropixels_local.yaml'
training_runtime_config = repo_root / 'colab_runtime_experiment.yaml'
checkpoint_dir = repo_root / 'artifacts' / 'checkpoints'
summary_path = repo_root / 'artifacts' / 'training_summary.json'

restored_training_export = restore_latest_exported_artifacts(
    drive_export_root=drive_export_root,
    local_artifact_root=repo_root / 'artifacts',
    runtime_experiment_config=training_runtime_config,
    training_run_id=TRAINING_RUN_ID,
)
if restored_training_export is None:
    raise FileNotFoundError(
        'No exported training runs were found under Drive. Run the training notebook first so discovery can restore a selected run_id.'
    )

selected_training_run_id = restored_training_export.parent.parent.name
discovery_export_timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
drive_discovery_export_path = build_notebook_discovery_export_path(
    drive_export_root=drive_export_root,
    run_id=selected_training_run_id,
    decode_type=DECODE_TYPE,
    attempt_timestamp=discovery_export_timestamp,
)

training_runtime_payload = yaml.safe_load(training_runtime_config.read_text(encoding='utf-8'))
runtime_subset_payload = training_runtime_payload.get('runtime_subset') or {}
split_manifest_path = Path(runtime_subset_payload.get('split_manifest_path', drive_data_root / 'splits' / 'split_manifest.json'))
split_counts = load_notebook_split_counts(
    split_manifest_path=split_manifest_path,
)
resolved_session_holdout_seed = (
    int(SESSION_HOLDOUT_SEED)
    if SESSION_HOLDOUT_SEED is not None
    else int(VALIDATION_SHUFFLE_SEED if VALIDATION_SHUFFLE_SEED is not None else training_runtime_payload.get('seed', 7))
)
inspect_arm = str(INSPECT_ARM).strip()
assert inspect_arm in {'baseline', 'whitening_only', 'whitening_plus_held_out_alignment'}, (
    "INSPECT_ARM must be one of 'baseline', 'whitening_only', or 'whitening_plus_held_out_alignment'."
)

discovery_split_name = str((training_runtime_payload.get('splits') or {}).get('discovery', 'discovery'))
test_split_name = str((training_runtime_payload.get('splits') or {}).get('test', 'test'))
staged_dataset = None
if STAGE_PREPARED_SESSIONS_LOCALLY:
    split_manifest = load_split_manifest(split_manifest_path)
    session_ids_to_stage = sorted(
        {
            assignment.recording_id.split('/', 1)[1]
            for assignment in split_manifest.assignments
            if assignment.split in {discovery_split_name, test_split_name}
        }
    )
    staged_dataset = materialize_notebook_prepared_sessions(
        source_dataset_root=drive_data_root,
        target_dataset_root=repo_dataset_root,
        session_ids=session_ids_to_stage,
        dataset_id=str(training_runtime_payload.get('dataset_id', 'allen_visual_behavior_neuropixels')),
    )

selected_checkpoint = resolve_notebook_checkpoint(summary_path=summary_path, checkpoint_dir=checkpoint_dir)
comparison_paths = build_notebook_discovery_comparison_paths(
    local_artifact_root=repo_root / 'artifacts',
    checkpoint_path=selected_checkpoint,
    decode_type=DECODE_TYPE,
)
runtime_experiment_config = comparison_paths.shared_runtime_experiment_config_path
experiment_config = build_notebook_discovery_runtime_config(
    source_experiment_config=training_runtime_config,
    runtime_experiment_config=runtime_experiment_config,
    artifact_root=repo_root / 'artifacts',
    decode_type=DECODE_TYPE,
    device_mode=DISCOVERY_DEVICE_MODE,
    step_log_every=NOTEBOOK_STEP_LOG_EVERY,
    discovery_max_batches=DISCOVERY_MAX_BATCHES,
    discovery_top_k_candidates=DISCOVERY_TOP_K_CANDIDATES,
    discovery_candidate_session_balance_fraction=DISCOVERY_CANDIDATE_SESSION_BALANCE_FRACTION,
    discovery_min_candidate_score=DISCOVERY_MIN_CANDIDATE_SCORE,
    discovery_min_cluster_size=DISCOVERY_MIN_CLUSTER_SIZE,
    discovery_probe_epochs=DISCOVERY_PROBE_EPOCHS,
    discovery_probe_learning_rate=DISCOVERY_PROBE_LEARNING_RATE,
    validation_max_batches=VALIDATION_MAX_BATCHES,
    validation_shuffle_seed=VALIDATION_SHUFFLE_SEED,
)
compute_targets = describe_notebook_compute_targets(experiment_config_path=experiment_config)

print('Requested training run ID:', TRAINING_RUN_ID or 'latest exported run')
print('Selected training run ID:', selected_training_run_id)
print('Restored training artifacts from:', restored_training_export)
print(f'Training runtime config: {training_runtime_config}')
print(f'Shared comparison runtime config: {experiment_config}')
print(f'Decode type: {DECODE_TYPE}')
print(
    f"Device mode: {DISCOVERY_DEVICE_MODE} | "
    f"encoder={compute_targets['encoder_device']} | "
    f"probe={compute_targets['probe_device']} | "
    f"clustering={compute_targets['clustering_device']} | "
    f"metrics={compute_targets['metrics_device']}"
)
print(f'Runtime split manifest: {split_manifest_path} | checkpoint: {selected_checkpoint.name}')
print(f'Realized split counts: {split_counts}')
print(
    f'Primary comparison: within-session held-out windows | fraction={SESSION_HOLDOUT_FRACTION} | '
    f'seed={resolved_session_holdout_seed} | inspect_arm={inspect_arm}'
)
print(f'Standard cross-session test validation enabled: {RUN_STANDARD_TEST_VALIDATION}')
if staged_dataset is not None:
    print('Prepared-session staging: local copy enabled')
    print(f'Staged sessions locally: {len(staged_dataset.staged_session_ids)} | target={staged_dataset.target_prepared_root}')
else:
    print('Prepared-session staging: using Drive-mounted dataset tree')
print('Local comparison root:', comparison_paths.comparison_root)
print('Drive discovery export path:', drive_discovery_export_path)

In [ ]:
paths_ok = verify_paths_exist({
    'shared_runtime_experiment_config': experiment_config,
    'data_config': data_config,
    'checkpoint': selected_checkpoint,
    'drive_dataset_root': drive_data_root,
})
print(paths_ok)
assert all(paths_ok.values()), 'Missing discovery comparison inputs.'

print('Using runtime subset described by the training notebook artifacts.')
print('Realized split counts:', split_counts)
print('Comparison arms: baseline, whitening_only, whitening_plus_held_out_alignment')
print('Primary metric family: within-session held-out probe + motif similarity')
print('Selected inspection arm:', inspect_arm)
print('Grouped comparison export path:', drive_discovery_export_path)

comparison_run = None
selected_arm_result = None

In [ ]:
reporter.begin('comparison', next_artifact='shared coverage summary + per-arm discovery artifacts + combined comparison summary')
%cd {repo_root}

comparison_run = run_notebook_discovery_representation_comparison(
    experiment_config_path=experiment_config,
    data_config_path=data_config,
    checkpoint_path=selected_checkpoint,
    local_artifact_root=repo_artifacts_root,
    decode_type=DECODE_TYPE,
    session_holdout_fraction=SESSION_HOLDOUT_FRACTION,
    session_holdout_seed=resolved_session_holdout_seed,
    run_standard_test_validation=RUN_STANDARD_TEST_VALIDATION,
    split_name='discovery',
    progress_ui=True,
)
selected_arm_result = getattr(comparison_run, inspect_arm)
decode_coverage_json = comparison_run.decode_coverage_summary_path
comparison_summary_json = comparison_run.comparison_summary_json_path
comparison_summary_csv = comparison_run.comparison_summary_csv_path
comparison_metadata_json = comparison_run.comparison_metadata_json_path

selected_discovery_run = selected_arm_result.discovery_run
selected_validation_run = selected_arm_result.validation_run
selected_discovery_artifact = selected_discovery_run.discovery_artifact_path
selected_cluster_summary_json = selected_discovery_run.cluster_summary_json_path
selected_cluster_summary_csv = selected_discovery_run.cluster_summary_csv_path
selected_validation_json = selected_validation_run.validation_summary_json_path
selected_validation_csv = selected_validation_run.validation_summary_csv_path
selected_transform_json = selected_arm_result.transform_summary_json_path
selected_transform_csv = selected_arm_result.transform_summary_csv_path

comparison_df = pd.read_csv(comparison_summary_csv)
print('Combined comparison summary:', comparison_summary_csv)
display(comparison_df)
for row in comparison_df.to_dict(orient='records'):
    print(
        f"{row['arm_name']}: discovery={row['discovery_probe_accuracy']:.3f} | "
        f"shuffled={row['shuffled_probe_accuracy']:.3f} | "
        f"primary held-out acc={row['primary_within_session_held_out_probe_accuracy']:.3f} | "
        f"primary motif ROC-AUC={row['primary_within_session_held_out_similarity_roc_auc']:.3f}"
    )
print('Selected inspection arm:', inspect_arm)
print('Selected discovery artifact:', selected_discovery_artifact)
print('Selected validation summary:', selected_validation_json)
print('Selected transform summary:', selected_transform_json)
reporter.finish('comparison')

In [ ]:
print('Inspection arm:', inspect_arm)
print('Shared decode coverage summary:', decode_coverage_json)
print('Combined comparison summary JSON:', comparison_summary_json)
print('Combined comparison summary CSV:', comparison_summary_csv)
print('Selected discovery artifact:', selected_discovery_artifact)
print('Selected cluster summary JSON:', selected_cluster_summary_json)
print('Selected cluster summary CSV:', selected_cluster_summary_csv)
print('Selected validation summary JSON:', selected_validation_json)
print('Selected validation summary CSV:', selected_validation_csv)
print('Selected transform summary JSON:', selected_transform_json)
print('Selected transform summary CSV:', selected_transform_csv)

## Results

Visualization and summary of the completed run.


In [ ]:
import json
import pandas as pd

with open(decode_coverage_json, 'r', encoding='utf-8') as fh:
    decode_coverage_payload = json.load(fh)
with open(selected_cluster_summary_json, 'r', encoding='utf-8') as fh:
    cluster_summary_payload = json.load(fh)
with open(selected_validation_json, 'r', encoding='utf-8') as fh:
    validation_payload = json.load(fh)
with open(selected_transform_json, 'r', encoding='utf-8') as fh:
    transform_payload = json.load(fh)

print('=== Shared Decode Coverage Summary ===')
print(
    f"  split={decode_coverage_payload['split_name']}  "
    f"target={decode_coverage_payload['target_label']}  "
    f"scanned={decode_coverage_payload['total_scanned_windows']}  "
    f"positive={decode_coverage_payload['positive_window_count']}  "
    f"negative={decode_coverage_payload['negative_window_count']}  "
    f"selected_positive={decode_coverage_payload['selected_positive_count']}  "
    f"selected_negative={decode_coverage_payload['selected_negative_count']}"
)
print(f"  positive-window sessions: {decode_coverage_payload['sessions_with_positive_windows']}")

print('\n=== Combined Arm Comparison ===')
comparison_df = pd.read_csv(comparison_summary_csv)
display(comparison_df)

print(f'\n=== Selected Arm Detail: {inspect_arm} ===')
cluster_df = pd.read_csv(selected_cluster_summary_csv)
display(cluster_df)

print(
    f"\nTotal clusters: {cluster_summary_payload['cluster_count']}  "
    f"Total candidates: {cluster_summary_payload['candidate_count']}"
)
for cluster in cluster_summary_payload.get('clusters', []):
    top_regions = ', '.join(r['value'] for r in cluster.get('top_regions', [])[:3]) or 'n/a'
    print(
        f"  Cluster {cluster['cluster_id']:>3}: "
        f"{cluster['candidate_count']} candidates, "
        f"{cluster['session_count']} sessions, "
        f"mean score {cluster['mean_score']:.3f}, "
        f"top regions: {top_regions}"
    )

print('\n=== Selected Arm Validation Summary ===')
def _coerce_metric(value):
    return float('nan') if value is None else value

fit_acc = _coerce_metric(validation_payload.get('discovery_fit_metrics', {}).get('probe_accuracy', float('nan')))
shuf_acc = _coerce_metric(validation_payload.get('shuffled_fit_metrics', {}).get('probe_accuracy', float('nan')))
primary_acc = _coerce_metric(validation_payload.get('primary_held_out_metrics', {}).get('probe_accuracy', float('nan')))
primary_auc = _coerce_metric(validation_payload.get('primary_held_out_similarity_summary', {}).get('window_roc_auc', float('nan')))
primary_pr = _coerce_metric(validation_payload.get('primary_held_out_similarity_summary', {}).get('window_pr_auc', float('nan')))
_sil = validation_payload.get('cluster_quality_summary', {}).get('silhouette_score')
_per = validation_payload.get('cluster_quality_summary', {}).get('cluster_persistence_mean')
sil_str = f'{_sil:.3f}' if _sil is not None else 'n/a'
persist_str = f'{_per:.3f}' if _per is not None else 'n/a'
status = validation_payload.get('comparison_status', 'ok')
failure_reason = validation_payload.get('failure_reason')
candidate_selection = validation_payload.get('candidate_selection_summary', {})
print(f'  Comparison status: {status}')
if failure_reason:
    print(f'  Failure reason: {failure_reason}')
print(
    f"  Candidate selection fallback used: {candidate_selection.get('fallback_used', False)} | "
    f"effective_min_score={candidate_selection.get('effective_min_score', 'n/a')}"
)
print(f'  Discovery fit accuracy: {fit_acc:.3f} | shuffled fit accuracy: {shuf_acc:.3f}')
print(f'  Primary within-session held-out accuracy: {primary_acc:.3f}')
print(f'  Primary within-session motif similarity: ROC-AUC={primary_auc:.3f}  PR-AUC={primary_pr:.3f}')
print(f'  Cluster quality: silhouette={sil_str}  persistence_mean={persist_str}')
standard_validation = validation_payload.get('standard_test_validation') or {}
if standard_validation:
    standard_acc = _coerce_metric(standard_validation.get('held_out_test_metrics', {}).get('probe_accuracy', float('nan')))
    standard_auc = _coerce_metric(standard_validation.get('held_out_similarity_summary', {}).get('window_roc_auc', float('nan')))
    standard_pr = _coerce_metric(standard_validation.get('held_out_similarity_summary', {}).get('window_pr_auc', float('nan')))
    print(f'  Secondary cross-session test accuracy: {standard_acc:.3f}')
    print(f'  Secondary cross-session motif similarity: ROC-AUC={standard_auc:.3f}  PR-AUC={standard_pr:.3f}')
print(f"  Excluded sessions from the primary held-out benchmark: {len(validation_payload.get('excluded_sessions', []))}")

print('\n=== Selected Arm Transform Summary ===')
display(pd.read_csv(selected_transform_csv))

In [ ]:
import json
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from collections import Counter

comparison_df = pd.read_csv(comparison_summary_csv)
with open(selected_cluster_summary_json, 'r', encoding='utf-8') as fh:
    cluster_payload = json.load(fh)
clusters = cluster_payload.get('clusters', [])

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Panel 1: Discovery fit vs shuffled fit across arms
x = range(len(comparison_df))
bar_width = 0.35
axes[0].bar([i - (bar_width / 2) for i in x], comparison_df['discovery_probe_accuracy'], width=bar_width, color='#55a868', label='Discovery fit')
axes[0].bar([i + (bar_width / 2) for i in x], comparison_df['shuffled_probe_accuracy'], width=bar_width, color='#dd8452', label='Shuffled fit')
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(comparison_df['arm_name'], rotation=15, ha='right')
axes[0].set_ylim(0, 1.05)
axes[0].set_title('Discovery Fit Accuracy By Arm')
axes[0].set_ylabel('Accuracy')
axes[0].legend(fontsize=8)

# Panel 2: Primary within-session held-out metrics across arms
axes[1].bar(comparison_df['arm_name'], comparison_df['primary_within_session_held_out_probe_accuracy'], color='#4c72b0', label='Held-out probe acc')
axes[1].plot(comparison_df['arm_name'], comparison_df['primary_within_session_held_out_similarity_roc_auc'], color='#c44e52', marker='o', linewidth=2, label='Held-out motif ROC-AUC')
axes[1].axhline(0.5, color='gray', linestyle='--', linewidth=0.8)
axes[1].set_ylim(0, 1.05)
axes[1].set_title('Primary Within-Session Held-out Metrics')
axes[1].set_ylabel('Score')
axes[1].tick_params(axis='x', rotation=15)
axes[1].legend(fontsize=8)

# Panel 3: Top brain regions for the selected inspection arm
region_counter = Counter()
for cluster in clusters:
    for entry in cluster.get('top_regions', []):
        region_counter[entry['value']] += entry['count']

if region_counter:
    top_regions = region_counter.most_common(10)
    names  = [r[0] for r in reversed(top_regions)]
    counts = [r[1] for r in reversed(top_regions)]
    axes[2].barh(names, counts, color='#8172b2')
    axes[2].set_title(f'Top Brain Regions ({inspect_arm})')
    axes[2].set_xlabel('Cumulative Candidate Count')
    for i, v in enumerate(counts):
        axes[2].text(v + 0.1, i, str(v), va='center', fontsize=8)
else:
    axes[2].text(0.5, 0.5, 'No region data', ha='center', va='center',
                 transform=axes[2].transAxes, fontsize=11, color='gray')
    axes[2].axis('off')

fig.suptitle('Discovery Representation Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
comparison_df = pd.read_csv(comparison_summary_csv)
best_row = comparison_df.sort_values(
    by='primary_within_session_held_out_probe_accuracy',
    ascending=False,
).iloc[0]
selected_row = comparison_df[comparison_df['arm_name'] == inspect_arm].iloc[0]

best_arm = best_row['arm_name']
best_acc = best_row['primary_within_session_held_out_probe_accuracy']
best_auc = best_row['primary_within_session_held_out_similarity_roc_auc']
selected_candidates = selected_row['candidate_count']
selected_clusters = selected_row['cluster_count']
selected_discovery_acc = selected_row['discovery_probe_accuracy']
selected_shuffled_acc = selected_row['shuffled_probe_accuracy']
selected_primary_acc = selected_row['primary_within_session_held_out_probe_accuracy']
selected_primary_auc = selected_row['primary_within_session_held_out_similarity_roc_auc']
selected_standard_acc = selected_row.get('standard_test_probe_accuracy')
selected_fallback_used = selected_row.get('candidate_selection_fallback_used')

standard_line = (
    f" Secondary cross-session test accuracy for this arm was {selected_standard_acc:.3f}."
    if pd.notna(selected_standard_acc)
    else ""
)
print(
    f"This notebook compared three representation arms for '{DECODE_TYPE}' on the same selected discovery windows, "
    f"using a within-session held-out benchmark as the primary comparison. "
    f"The best primary held-out probe accuracy came from {best_arm} at {best_acc:.3f}, "
    f"with primary motif ROC-AUC {best_auc:.3f}.\n\n"
    f"The selected inspection arm ({inspect_arm}) produced {int(selected_clusters)} non-noise cluster(s) from "
    f"{int(selected_candidates)} candidates. Its discovery-fit probe accuracy was {selected_discovery_acc:.3f} "
    f"versus shuffled-fit accuracy {selected_shuffled_acc:.3f}, and its primary held-out probe accuracy was "
    f"{selected_primary_acc:.3f} with primary motif ROC-AUC {selected_primary_auc:.3f}. "
    f"Candidate fallback used: {selected_fallback_used}."
    f"{standard_line}"
)

In [ ]:
reporter.begin('export', next_artifact='Drive copy of the current discovery attempt')
drive_discovery_export_path = export_notebook_discovery_comparison_artifacts(
    drive_export_root=drive_export_root,
    local_artifact_root=repo_artifacts_root,
    run_id=selected_training_run_id,
    decode_type=DECODE_TYPE,
    attempt_timestamp=discovery_export_timestamp,
)
print('Exported grouped comparison attempt to:', drive_discovery_export_path)
reporter.finish('export')